# Deep Research 104 — Demo Notebook

End-to-end walkthrough of the Deep Research pipeline using a stub model
(no API keys required). Updated 17 October 2026.

## 1. Setup

Install the package in editable mode if you haven't already:

```bash
pip install -e .[dev]
```

In [ ]:
import asyncio
from src.models.base_model import BaseModel, GenerationConfig, ModelResponse, ModelCapability
from src.pipeline.deep_research import DeepResearchPipeline
from config import __version__, __build_date__

print(f'Deep Research 104 v{__version__} (build {__build_date__})')

## 2. Stub model

A `BaseModel` subclass that returns canned responses so we can exercise
the pipeline without an API key.

In [ ]:
class StubModel(BaseModel):
    def __init__(self, canned='Fusion energy breakthroughs in 2026 include ITER milestone.'):
        super().__init__('stub-1')
        self.canned = canned
        self.capabilities = {ModelCapability.STREAMING, ModelCapability.REASONING}

    async def generate(self, messages, config=None):
        text = ' '.join(m.get('content', '') for m in messages)
        return ModelResponse(
            content=self.canned,
            model=self.model_name,
            prompt_tokens=max(1, len(text) // 4),
            completion_tokens=max(1, len(self.canned) // 4),
            total_tokens=max(2, (len(text) + len(self.canned)) // 4),
            latency_ms=10.0,
        )

    async def stream_generate(self, messages, config=None):
        for word in self.canned.split():
            yield word + ' '

    def count_tokens(self, text):
        return max(1, len(text) // 4)

    def get_context_limit(self):
        return 16384

model = StubModel()
print(f'Stub model: {model}')

## 3. Run the pipeline

Build a `DeepResearchPipeline` with `max_iterations=1` for a fast demo.
The pipeline will:
1. Generate search queries from the question
2. Search the web (may fail without network — that's OK)
3. Synthesize a final answer

In [ ]:
pipeline = DeepResearchPipeline(model=model, max_iterations=1)
report = await pipeline.research('What are the latest fusion breakthroughs?', fast_mode=True)
print(report.to_markdown())
print()
print(f'Tokens used: {report.total_tokens_used}')
print(f'Cost: ${report.total_cost_usd:.6f}')

## 4. Run with a real provider

Set `OPENAI_API_KEY` or `DEEPSEEK_API_KEY` in your environment, then re-run
with `get_model_factory`:

In [ ]:
import os
if os.environ.get('OPENAI_API_KEY'):
    from src.models import get_model_factory
    real_model = get_model_factory('openai')
    pipeline = DeepResearchPipeline(model=real_model, max_iterations=2)
    report = await pipeline.research('What is retrieval-augmented generation?')
    print(report.to_markdown())
    await real_model.close()
else:
    print('Set OPENAI_API_KEY to run this cell against the real OpenAI API.')